# Consistent condition

The story is split into blocks (a line with `###` between them). In every block **one fixed substitution per phoneme** is applied to **every** occurrence, e.g. block 1: every t → tʲ, every k → x, every e → ø.
For every block you choose the **voice**, the **errors** and the **reference voice** whose timing (reading the original text) is used.

The output is written to disk (an HTML player and a wav per block and for the whole condition); nothing is displayed in the notebook.

All functions are in `accent_tools.py` (next to `functions.py`); `help(make_consistent)`, `help(render_condition)` … show their documentation.

In [1]:
from accent_tools import *
from pathlib import Path
import torch
VOICES_DIR = Path("custom_voices")


c:\Users\utente\anaconda3\envs\koko\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\utente\anaconda3\envs\koko\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
c:\Users\utente\anaconda3\envs\koko\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


## Settings
Everything you need to change is in this cell. `BLOCKS` has one line per block, in story order.

In [5]:
# ---- story --------------------------------------------------------------------------------------------
STORY_FILE = "story1.txt"     # blocks separated by a line containing only ###
STORY_NAME = None             # name used in titles and file names (None = file name, e.g. "story1")
BLOCK_SEP  = "###"

# ---- one line per block: voice, reference voice (timing), errors {native: replacement} ----------------
#   ref_voice=None : that block keeps its own timing;  errors={} : that block stays native
#   ɡ = U+0261, the g in Kokoro's vocab (not the keyboard g)

vb1 = torch.load(VOICES_DIR / "nicola_20.pt",  weights_only=True)
vb2 = torch.load(VOICES_DIR / "nicola_50.pt",    weights_only=True)
vb3 = torch.load(VOICES_DIR / "nicola_80.pt",      weights_only=True)
BLOCKS = [
    dict(voice=vb1, ref_voice="im_nicola", errors={"t": "θ",  "k": "ɡ",  "i": "y"}),   # block 1
    dict(voice=vb2,   ref_voice="im_nicola", errors={"t": "tʲ", "k": "x",  "e": "ø"}),   # block 2
    dict(voice=vb3, ref_voice="im_nicola", errors={"t": "ɖ",  "k": "kʲ", "o": "ə"}),   # block 3
]

# ---- timing and prosody --------------------------------------------------------------------------
# The reference voice of each block (ref_voice above) reads the ORIGINAL text; what is taken from it:
TRANSFER = dict(duration=True,   # take the reference voice's durations
                pitch=False,     # take its pitch contour too (moved into the block voice's range)
                energy=False)    # take its energy contour too
DUR_MODE = "raw"     # "raw":  exactly the reference voice's frames -> identical timing for every voice/condition
                     # "rate": the reference rhythm, stretched to each block voice's own speaking rate
# In a palatalized consonant (kʲ, tʲ ...) the consonant gets the reference duration and the added ʲ keeps
# the block voice's own duration (KEEP_OWN). Add "ʰ" to treat aspiration the same way, or () to split.
KEEP_OWN = ("ʲ",)

# ---- measurement (printed in the summary and saved in the _summary.csv) -----------------------------
# weight of each substitution for the weighted distance pw_w; pairs not listed = number of features
WEIGHTS = {("t", "tʲ"): 1, ("t", "θ"): 3, ("t", "ɖ"): 2,
           ("k", "x"):  3, ("k", "ɡ"): 2, ("k", "kʲ"): 1,
           ("e", "ø"):  1.5, ("i", "y"): 1.5, ("o", "ə"):  3}

# ---- output -------------------------------------------------------------------------------------------
OUT_DIR       = "output_online/consistent"
EXPORT_NATIVE = False                       # also export a native version with the same voices and timing
NATIVE_DIR    = "output_online/native"
N_PEEK        = 2                           # sentences per block shown in the IPA preview

BLOCK_VOICES = [b["voice"] for b in BLOCKS]
REF_VOICE    = [b.get("ref_voice") for b in BLOCKS]
ERRORS       = [b.get("errors", {}) for b in BLOCKS]

## 1. Load the story
Splits the story into blocks at the `###` lines, converts every sentence to IPA and turns every ɪ into i.
The printout shows the number of sentences and words per block, check that it matches your blocks.

In [6]:
story = load_story(STORY_FILE, name=STORY_NAME, block_sep=BLOCK_SEP)

story1: 3 block(s), block 1: 35 sentences / 733 words, block 2: 35 sentences / 735 words, block 3: 36 sentences / 740 words   (IPA clean-up: 230 × ɪ→i)


## 2. Manipulate
Every occurrence of each phoneme in `errors` is replaced in its block. The printout shows how many were replaced; the summary table shows, per block, what changed and how much (`pw` = per-word normalized distance, every change = 1; `pw_w` = with your `WEIGHTS`; both in %, see `help(measure_distance)`).

In [7]:
cond = make_consistent(story, errors=ERRORS)
print_summary(cond, weights=WEIGHTS);

consistent (story1), block 1: t→θ ×216, k→ɡ ×116, i→y ×236
consistent (story1), block 2: t→tʲ ×202, k→x ×163, e→ø ×267
consistent (story1), block 3: t→ɖ ×215, k→kʲ ×127, o→ə ×304

consistent · story1
  block  sent  words  changed    pw %  pw_w %   changes
  1        35    733      402    16.4    33.2   i→y ×236, k→ɡ ×116, t→θ ×216
  2        35    735      435    20.7    36.3   e→ø ×267, k→x ×163, t→tʲ ×202
  3        36    740      363    14.9    33.5   k→kʲ ×127, o→ə ×304, t→ɖ ×215
  all     106   2208     1200    17.3    34.3   


### Look at the accented IPA (optional)
The first sentences of every block, native above accented. All sentences are also saved in `_ipa.txt` at export.

In [8]:
for blk in cond["blocks"]:
    print(f"── block {blk['index']}")
    for nat, man in list(zip(blk["ipa"], blk["manip"]))[:N_PEEK]:
        print("   native  :", nat)
        print("   accented:", man)

── block 1
   native  : irˈɛne vˈalli lavorˈava kˌome botˈanika allʊniversitˈa di bolˈoɲɲa, e da mˈolti ˈanni stʊdjˈava le pjˈante ke krˈeskono viʧˈino al mˈare.
   accented: yrˈɛne vˈally lavorˈava ɡˌome boθˈanyɡa allʊnyversyθˈa dy bolˈoɲɲa, e da mˈolθy ˈanny sθʊdjˈava le pjˈanθe ɡe ɡrˈesɡono vyʧˈyno al mˈare.
   native  : kwellˈanno dovˈɛva rakːˈɔʎere alkˌʊni kampjˈoni di ˌʊna pˈikːola pjˈanta kon fjˈori ʤˈalli, ke kreʃˈɛva soltˈanto sˌʊlle rˈɔʧːe di ʊnˈizola del sˈud.
   accented: ɡwellˈanno dovˈɛva raɡːˈɔʎere alɡˌʊny ɡampjˈony dy ˌʊna pˈyɡːola pjˈanθa ɡon fjˈory ʤˈally, ɡe ɡreʃˈɛva solθˈanθo sˌʊlle rˈɔʧːe dy ʊnˈyzola del sˈud.
── block 2
   native  : lˈaːʊtobʊs ˌɛra pˈikːolo e blˈu, e ˌalle ˈɔtːo e ʊn kwˈarto si e fermˈato davˈanti al pˈɔrto kon ˌʊna preʧiziˈone ke irˈɛne non si aspetːˈava.
   accented: lˈaːʊtʲobʊs ˌɛra pˈixːolo ø blˈu, ø ˌallø ˈɔtʲːo ø ʊn xwˈartʲo si ø førmˈatʲo davˈantʲi al pˈɔrtʲo xon ˌʊna prøʧiziˈonø xø irˈɛnø non si aspøtʲːˈava.
   native  : a bˈɔrdo ʧˈɛrano s

## 3. Synthesize
One call per block. This is the slow step (a few seconds per sentence on CPU).

In [9]:
results = render_condition(cond, voices=BLOCK_VOICES, ref_voice=REF_VOICE, **TRANSFER,
                           dur_mode=DUR_MODE, keep_own=KEEP_OWN)

  consistent block 1: voice voice0, duration from im_nicola reading the original text


  consistent block 2: voice voice0, duration from im_nicola reading the original text
  consistent block 3: voice voice0, duration from im_nicola reading the original text


## 4. Export
Writes everything to `OUT_DIR`; nothing is displayed here. Open the `.html` files in a browser.

| file | content |
|---|---|
| `<condition>_<story>_block1.html` … | interactive player of one block (word boxes, token durations, altered phonemes highlighted, pitch / energy panel, legend of the changes in that block) |
| `<condition>_<story>_block1.wav` … | audio of one block |
| `<condition>_<story>_full.html` / `.wav` | the whole condition, blocks one after the other |
| `<condition>_<story>_ipa.txt` | native and accented IPA of every sentence |
| `<condition>_<story>_summary.csv` | per block: voice, reference voice, duration, words, changes, distances |

In [10]:
export_condition(cond, results, OUT_DIR, weights=WEIGHTS);

written to output_online\consistent/ :
    consistent_story1_block1.html
    consistent_story1_block1.wav
    consistent_story1_block2.html
    consistent_story1_block2.wav
    consistent_story1_block3.html
    consistent_story1_block3.wav
    consistent_story1_full.html
    consistent_story1_full.wav
    consistent_story1_ipa.txt
    consistent_story1_summary.csv


## 5. (Optional) Native version with the same voices and timing
The same story without any error, with the same block voices and the same reference timing, so it can be compared with the accented condition one to one.

In [11]:
if EXPORT_NATIVE:
    native = make_native(story)
    native_results = render_condition(native, voices=BLOCK_VOICES, ref_voice=REF_VOICE, **TRANSFER,
                                      dur_mode=DUR_MODE, keep_own=KEEP_OWN)
    export_condition(native, native_results, NATIVE_DIR);